# AIMA Skin Lesion Segmentation — canonical Kaggle workflow

This thin notebook copies the attached read-only source into `/kaggle/working`, installs the tested package, and invokes the same CLI used by CI. Run one model/seed training job per Kaggle session. Do not enable comparison until all six runs are complete.

In [ ]:
from pathlib import Path
import json
import re
import shutil
import subprocess
import sys
import zipfile

KAGGLE_INPUT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working/aima-skin-lesion-runtime')
SOURCE_ARCHIVE_HINT = None  # Set to the attached source ZIP if discovery is ambiguous.
SOURCE_COMMIT = ''  # Paste the full 40-character commit used to create the source ZIP.
DATASET_ROOT = Path('/kaggle/input/REPLACE_WITH_AUTHORISED_DATASET')

RUN_VERIFY = True
RUN_PREPARE = False
RUN_TRAINING = False
RUN_COMPARISON = False
RUN_SUBMISSION = False
RLE_ORDER_CONFIRMED = False

MODEL = 'unet'  # exactly 'unet' or 'attention_unet'
SEED = 42       # exactly 42, 43, or 44
assert MODEL in {'unet', 'attention_unet'}
assert SEED in {42, 43, 44}


In [ ]:
def safe_extract(archive: Path, destination: Path) -> None:
    with zipfile.ZipFile(archive) as handle:
        root = destination.resolve()
        for member in handle.infolist():
            target = (destination / member.filename).resolve()
            if root not in target.parents and target != root:
                raise ValueError(f'Unsafe archive member: {member.filename}')
        handle.extractall(destination)

if SOURCE_ARCHIVE_HINT is None:
    archives = sorted(KAGGLE_INPUT.rglob('*.zip'))
    matches = [path for path in archives if 'skin-lesion' in path.name.lower()]
    if len(matches) != 1:
        raise RuntimeError(f'Set SOURCE_ARCHIVE_HINT; candidates={matches}')
    source_archive = matches[0]
else:
    source_archive = Path(SOURCE_ARCHIVE_HINT)

if WORKING_ROOT.exists():
    shutil.rmtree(WORKING_ROOT)
WORKING_ROOT.mkdir(parents=True)
safe_extract(source_archive, WORKING_ROOT)
roots = [path.parent for path in WORKING_ROOT.rglob('pyproject.toml')]
if len(roots) != 1:
    raise RuntimeError(f'Expected one repository root, found {roots}')
REPO_ROOT = roots[0]
if re.fullmatch(r'[0-9a-fA-F]{40}', SOURCE_COMMIT) is None:
    raise ValueError('SOURCE_COMMIT must be the full commit used for this archive')
(REPO_ROOT / 'SOURCE_COMMIT').write_text(SOURCE_COMMIT.lower() + '\n', encoding='ascii')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(REPO_ROOT)], check=True)
print('Writable source:', REPO_ROOT)


In [ ]:
runtime_config = json.loads((REPO_ROOT / 'configs/model_comparison.json').read_text())
runtime_config.update({
    'image_dir': str(DATASET_ROOT / 'train/images'),
    'mask_dir': str(DATASET_ROOT / 'train/masks'),
    'group_mapping_path': None,
})
RUNTIME_CONFIG = WORKING_ROOT / 'model_comparison.runtime.json'
RUNTIME_CONFIG.write_text(json.dumps(runtime_config, indent=2, sort_keys=True) + '\n')
MANIFEST = WORKING_ROOT / 'comparison_manifest.json'
RUN_ROOT = WORKING_ROOT / 'runs' / f'{MODEL}-{SEED}'

def run_command(arguments):
    command = [sys.executable, '-m', 'skin_lesion_segmentation.cli', *map(str, arguments)]
    print(subprocess.list2cmdline(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
if RUN_VERIFY:
    subprocess.run([sys.executable, 'scripts/verify_repository.py'], cwd=REPO_ROOT, check=True)
    subprocess.run([sys.executable, '-m', 'pytest', '-q', '-W', 'error'], cwd=REPO_ROOT, check=True)
    run_command(['smoke', '--model', 'unet'])
    run_command(['smoke', '--model', 'attention_unet'])

if RUN_PREPARE:
    if MANIFEST.exists():
        raise FileExistsError('Refusing to replace the immutable manifest')
    run_command(['prepare-comparison', '--config', RUNTIME_CONFIG, '--output-manifest', MANIFEST])


In [ ]:
if RUN_TRAINING:
    if not MANIFEST.is_file():
        raise FileNotFoundError('Attach or prepare the one immutable comparison manifest')
    run_command([
        'train', '--config', RUNTIME_CONFIG,
        '--split-manifest', MANIFEST,
        '--model', MODEL,
        '--seed', SEED,
        '--output-dir', RUN_ROOT,
    ])


In [ ]:
UNET_RUN_ROOT = Path('/kaggle/input/REPLACE_UNET_RUN')
ATTENTION_RUN_ROOT = Path('/kaggle/input/REPLACE_ATTENTION_RUN')
COMPARISON_OUTPUT = WORKING_ROOT / 'comparisons' / f'seed-{SEED}'

if RUN_COMPARISON:
    run_command([
        'compare', '--config', RUNTIME_CONFIG,
        '--split-manifest', MANIFEST,
        '--unet-run-summary', UNET_RUN_ROOT / 'run_summary.json',
        '--attention-run-summary', ATTENTION_RUN_ROOT / 'run_summary.json',
        '--unet-checkpoint', UNET_RUN_ROOT / 'best_model.keras',
        '--attention-checkpoint', ATTENTION_RUN_ROOT / 'best_model.keras',
        '--output-dir', COMPARISON_OUTPUT,
        '--bootstrap-seed', 20260730 + SEED,
        '--bootstrap-resamples', 10000,
    ])

if RUN_SUBMISSION:
    if not RLE_ORDER_CONFIRMED:
        raise RuntimeError('Submission is blocked until the competition RLE contract is confirmed')
    raise RuntimeError('Use the documented controlled-rerun submit command; Submission is not part of model selection')
